# Interview concepts

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-14-interview-concepts.ipynb)

Interviewers rarely ask for a single query. They ask why a query behaves the way it does, what an index is for,
and which trade-off you would choose. This chapter answers the core concept questions in short, direct form. Each
answer says what to remember in the interview, and includes a runnable demo where Spark can show it.

The concepts are grouped into four areas: how queries are evaluated, data modeling, storage and performance, and
correctness traps. Practice questions with answers follow the list.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

## 1. How queries are evaluated

### Q: What does `EXPLAIN` show, and what should you look for?

**Answer.** The plan the engine chose: which tables it scans, which filters it pushes down, which join strategy it
uses, and where it moves data between machines. Look for full scans of large tables, `Exchange` nodes (shuffles),
and joins that are slower than they should be for their input sizes. `EXPLAIN FORMATTED` prints the plan with
details. Engines differ on `EXPLAIN ANALYZE`, which runs the query and reports actual row counts and timings.

### Where you'll use it

- **Slow dashboards:** the plan shows whether a filter runs before the join and whether a join scans a whole table. This is the first thing to read when a report times out.

### Where you'll use it

- **Slow-query triage:** `EXPLAIN` is the first tool an engineer opens when a dashboard times out.

In [ ]:
spark.sql("""
EXPLAIN FORMATTED
SELECT c.region, SUM(o.amount) AS total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
GROUP BY c.region
""").first()[0]

### Q: What is the difference between `WHERE` and `HAVING`, and between `GROUP BY` and a window function?

**Answer.** `WHERE` filters rows before grouping; `HAVING` filters groups after aggregation. `GROUP BY` returns one
row per group; a window function keeps every row and adds a computed column. The basics and window chapters show
both with examples.

### Q: `CTE` or subquery? Does a CTE make a query faster?

**Answer.** A CTE is for readability: it names a step so the query reads top to bottom. It does not guarantee
speed. Some engines compute a CTE once and store it; others inline it into each place it is used. Check the plan
before you assume either. The aggregations chapter shows CTEs, subqueries and a recursive CTE.

### Where you'll use it

- **Code review:** a dbt model written as chained CTEs is easier to test step by step than one deep nested subquery, even when the engine produces the same plan.

### Where you'll use it

- **Report logic:** `WHERE` removes rows before grouping (for example, only 2024 orders); `HAVING` filters the groups afterwards (for example, customers above 1,000 in spend).

## 2. Data modeling

### Q: What is normalisation? Explain 1NF, 2NF and 3NF, and when would you denormalise?

**Answer.**

| Form | Rule | Typical violation |
| --- | --- | --- |
| 1NF | every column holds one atomic value; no repeating groups | a `phones` column holding `"555-1, 555-2"` |
| 2NF | every non-key column depends on the whole key | in `(order_id, product_id)`, a `product_name` that depends only on `product_id` |
| 3NF | no non-key column depends on another non-key column | `customer_city` stored on orders, although it depends on `customer_id` |

Normalise transactional systems to avoid update anomalies. Denormalise analytics tables, such as a fact table
that carries the customer's region, when reads matter more than writes. Keep the grain clear either way.

The demo below shows the cost of a denormalised table: the customer's name is repeated on every order, so
counting customers needs `DISTINCT`.

### Where you'll use it

- **Operational systems:** an order system stores each customer once and refers to them by key, so a change of address is one update.
- **Analytics:** a warehouse often denormalises on purpose, so a dashboard does not repeat joins for every query.

### Where you'll use it

- **Choosing a model for a team:** normalised models for operational systems, star schemas for BI and analytics, and a deliberate denormalisation where a read path is hot.

In [ ]:
spark.sql("""
WITH denormalised AS (
  SELECT o.order_id, c.customer_id, c.name AS customer_name
  FROM orders o JOIN customers c ON c.customer_id = o.customer_id
)
SELECT COUNT(*) AS rows_, COUNT(DISTINCT customer_id) AS customers FROM denormalised
""").show()

### Q: Star schema or snowflake schema?

**Answer.** A star schema has one fact table joined directly to denormalised dimension tables. A snowflake schema
splits each dimension into normalised sub-tables, such as `product -> brand -> category`. Stars are simpler to query
and usually faster for analytics; snowflakes save storage and keep dimensions consistent. The modeling chapter
builds a star schema.

### Q: What is a slowly changing dimension? Type 1, 2 and 3?

**Answer.** A dimension attribute that changes over time.

- **Type 1** overwrites the old value. History is lost.
- **Type 2** adds a row per version, with validity dates. History is kept.
- **Type 3** adds a column for the previous value. Only one change of history is kept.

The modeling chapter implements Type 2 with half-open intervals.

### Where you'll use it

- **History reports:** a customer moves from Oslo to Lima. A Type 2 dimension keeps both rows, so last year's revenue still shows Oslo.

### Where you'll use it

- **BI tools:** Looker, Power BI and Tableau models are built as star schemas, because the flat dimensions give simpler, faster queries.

## 3. Storage and performance

### Q: What is an index? What does a B-tree index do?

**Answer.** An index is a separate structure that maps a key to the rows holding it. A B-tree is a balanced tree, so
a lookup takes a logarithmic number of steps instead of a scan of the whole table. Indexes speed up point lookups
and selective range filters, and they add cost to every write and use storage. Spark has no user-created indexes;
it relies on partitioning, file statistics and data skipping. Warehouses such as PostgreSQL, MySQL and SQL Server
expose indexes directly.

### Q: Clustered vs non-clustered index. What is a covering index?

**Answer.** A clustered index stores the table's rows in the index order, so there is one per table. A
non-clustered index is a separate structure that points to the rows. A **covering** index contains every column a
query needs, so the engine answers from the index and never reads the table.

### Q: What is partitioning and how does it help?

**Answer.** Partitioning splits a table into pieces by a column such as date. A query that filters on the partition
column reads only the matching pieces, which is called partition pruning. The risk is too many small partitions, or
a filter that wraps the partition column in a function and prevents pruning. The `INSERT OVERWRITE` statement in the
views chapter overwrites partitions in the same way.

### Q: How would you approach optimising a slow query?

**Answer.** Measure first. Then, in order:

1. Check the plan for full scans, large shuffles and the wrong join strategy.
2. Make filters sargable: compare bare columns with constants, and keep types consistent.
3. Reduce the data early: filter before joining, select only the columns needed, and avoid `SELECT *`.
4. Fix the join: broadcast a small table, and check for row multiplication and skew.
5. Change the layout last: partition or cluster on the column your queries filter by.

Re-run and compare the plan and the timing after each change.

### Where you'll use it

- **Triage in production:** check the plan and the row counts, fix the join or filter that multiplies data, then consider an index or partition.

### Where you'll use it

- **Time-based tables:** a fact table split by month lets a query for one month read one partition, and lets old partitions be dropped cheaply.

### Where you'll use it

- **Covering indexes:** an index that already holds every column a dashboard query needs, so the table itself is never read.

### Where you'll use it

- **Point lookups:** a B-tree index on `email` finds one customer in a large `customers` table without scanning it. OLTP databases such as PostgreSQL and MySQL rely on this.

### Where you'll use it

- **Index and partition decisions:** OLTP databases (PostgreSQL, MySQL) use indexes; warehouses use partitioning and clustering. The same question, two toolkits.

## 4. Correctness and transactions

### Q: What does ACID mean, and what are the isolation levels?

**Answer.** **Atomicity**: a transaction takes effect completely or not at all. **Consistency**: it moves the
database from one valid state to another. **Isolation**: concurrent transactions do not interfere in ways the level
forbids. **Durability**: committed changes survive a crash.

Isolation levels, from weakest to strongest, and the anomalies each allows:

| Level | Dirty read | Non-repeatable read | Phantom read |
| --- | --- | --- | --- |
| Read uncommitted | possible | possible | possible |
| Read committed | no | possible | possible |
| Repeatable read | no | no | possible |
| Serializable | no | no | no |

Warehouses and table formats such as Delta and Iceberg usually give snapshot isolation: each query reads a consistent
snapshot, and writes commit atomically. Spark's built-in tables and temporary views do not.

### Q: What is the difference between `DELETE`, `TRUNCATE` and `DROP`?

**Answer.** `DELETE` removes rows matching a condition and can be rolled back in a transaction. `TRUNCATE` removes
all rows, is usually faster, and often cannot be rolled back. `DROP` removes the table itself, including its
definition. Table formats may log `DELETE` as a new commit, which keeps history available for time travel.

### Q: Why does `NOT IN` return no rows when the list contains NULL? Why does `COUNT(*)` differ from `COUNT(column)`?

**Answer.** `x NOT IN (a, NULL)` is unknown for every x, so no row passes. `COUNT(*)` counts rows; `COUNT(column)`
counts non-NULL values. The NULL chapter demonstrates both.

### Q: `UNION` vs `UNION ALL`?

**Answer.** `UNION` removes duplicates, which costs a sort or hash step; `UNION ALL` keeps every row. The set
operations chapter shows the difference.

### Q: Where should a condition on the right-hand table of a left join go: `WHERE` or `ON`?

**Answer.** In `ON`. A condition in `WHERE` runs after the join and removes the unmatched rows, which turns the left
join into an inner join. A condition in `ON` decides which right rows can match, and keeps every left row. The
joins chapter shows the difference.

### Where you'll use it

- **Reports that must keep every customer:** a filter placed in `ON` keeps unmatched customers; the same filter in `WHERE` quietly turns the left join into an inner join.

### Where you'll use it

- **Combining daily tables:** `UNION ALL` stacks daily extracts quickly; `UNION` is only needed when duplicates must be removed.

### Where you'll use it

- **Anti-joins in pipelines:** finding unmatched records with `NOT EXISTS` is the safe default, because one NULL in the list silently empties the result.

### Where you'll use it

- **Refreshing a staging table:** `TRUNCATE` clears a load table quickly before reloading it; `DELETE` is used when only some rows must go and the change must be logged.

### Where you'll use it

- **Money movement:** a bank transfer must debit one account and credit another as one unit. ACID transactions guarantee that a failure leaves neither change applied.

### Where you'll use it

- **Data correctness:** isolation levels decide what a report sees while a load runs. Anti-join and NULL pitfalls cause wrong reports in production.

## 5. More questions interviewers ask

### Q: Why does `COUNT(*)` on an empty input return 0, while `MAX` returns NULL?

**Answer.** `COUNT` is defined to return 0 for no rows. `SUM`, `AVG`, `MIN` and `MAX` return NULL, because there is
no value to report. Wrap them in `COALESCE` when a report needs zero.

### Where you'll use it

- **Empty-day reports:** a dashboard shows zero sales for a day with no orders, not a missing value, because `COUNT` returns 0.

### Where you'll use it

- **Code review and performance reviews:** the sargability and `SELECT *` points are the first comments on many pull requests.

In [ ]:
spark.sql("""
SELECT COUNT(*) AS count_of_nothing, MAX(amount) AS max_of_nothing
FROM orders WHERE 1 = 0
""").show()

### Q: What is data skew and how do you spot it?

**Answer.** Skew means a few keys hold most of the rows. In a join or a `GROUP BY`, the machine that handles the hot
key does most of the work and everyone waits for it. Spot it by counting rows per key and comparing the largest
count with the average. Fixes include broadcasting the small side of a join (the modeling chapter shows the
`BROADCAST` hint), or salting the hot key, an advanced technique.

### Where you'll use it

- **Slow joins at scale:** one customer or one key holds most rows, so one task runs for hours while others finish in seconds. Salting or a skew hint fixes it.

In [ ]:
spark.sql("""
SELECT customer_id, COUNT(*) AS rows_per_key
FROM orders
GROUP BY customer_id
ORDER BY rows_per_key DESC
""").show()

### Q: Why avoid `SELECT *` in production?

**Answer.** Columnar storage reads only the columns a query names, so `SELECT *` reads and transfers everything. It
also breaks when the table gains or reorders columns, and it hides which columns a downstream consumer depends on.

### Q: Which filter is sargable: `WHERE YEAR(order_date) = 2023` or `WHERE order_date >= DATE '2023-01-01'`?

**Answer.** The second. The first wraps the column in a function, so the engine cannot use an index or prune
partitions on it. The date-functions section of the types chapter and the modeling chapter show the same point.

### Q: How do you find the second-highest salary, or the top N per group?

**Answer.** For the second-highest, use `MAX` below the maximum or `DENSE_RANK` equal to 2. For the top N per group,
use `ROW_NUMBER` in a subquery, with a deterministic tie-breaker. The interview drills, Part A, work through both.

## Practice questions

1. Why does `SELECT 0.1 + 0.2 = 0.3` differ between Spark `DECIMAL` literals and `DOUBLE` values? (Explain in one sentence.)
2. Write a query that returns the customers with no orders, using a method that is safe when the orders table can contain NULL customer ids. Show `customer_id`, sorted.
3. Write a query that shows, for each status, how many orders there are, and labels the output columns `status` and `orders`. Sort by status.

## Answers

### Where you'll use it

- **Top-N per group:** the top three products in each category is a standard analytics question, solved with `DENSE_RANK` or `ROW_NUMBER` over a partition.

### Where you'll use it

- **Index and partition use:** `order_date >= '2023-01-01'` can use an index or prune partitions; `YEAR(order_date) = 2023` usually cannot.

### Where you'll use it

- **Cost and stability:** on a pay-per-scan warehouse, `SELECT *` reads every column and breaks downstream models when a column is added.

### Question 1

Spark `DECIMAL` arithmetic is exact, so `0.1 + 0.2` is `0.3`. `DOUBLE` is binary floating point, where `0.1` and
`0.2` have no exact representation, so the sum is `0.30000000000000004`. The types chapter shows both.

### Question 2

In [ ]:
q = """
SELECT c.customer_id
FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(7,), (8,)]

`NOT EXISTS` is safe with NULLs in the subquery, where `NOT IN` is not. Its answer is the same here because
`orders.customer_id` has no NULLs.

### Question 3

In [ ]:
q = """
SELECT status, COUNT(*) AS orders
FROM orders
GROUP BY status
ORDER BY status
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("cancelled", 2), ("completed", 11), ("pending", 1)]